<a href="https://colab.research.google.com/github/xxmadara1xx/Controllo-LED-tramite-UART/blob/main/aerial_v5_final_ipynb_(1)_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛸 Aerial Trainer v5 — YOLO11m
### 4 كلاسات: drone | airplane | helicopter | bird
### شغّل الخلايا بالترتيب: 1 → 2 → 3 → 4 → 5

| Cell | المهمة |
|---|---|
| 1 | تثبيت المكتبات |
| 2 | ربط Google Drive |
| 3 | تحميل الداتا ودمجها |
| 4 | التحقق البصري من الداتا |
| 5 | التدريب مع حفظ تلقائي واستئناف |

> ⚠️ فعّل GPU قبل البداية: Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1: تثبيت
!pip install roboflow ultralytics -q

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ لا يوجد GPU'
print(f'الجهاز: {gpu}')
print('Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 118.3 MB/s eta 0:00:00
الجهاز: Tesla T4
Done!


In [ ]:
# Cell 2: Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive connected!')

Mounted at /content/drive
Drive connected!


In [ ]:
# Cell 3: تحميل الداتاسيتات ودمجها
#
# المصادر:
#   drones_new v4  (22K صورة) — drone
#   Airborne   v6  (22K صورة) — airplane + helicopter + bird + drone
#
# الكلاسات النهائية:
#   0 = drone  |  1 = airplane  |  2 = helicopter  |  3 = bird
#
# ملاحظة: format='yolov8' = YOLO txt files — يعمل مع YOLO11 تماماً ✅

import os, shutil, yaml
from roboflow import Roboflow
from pathlib import Path

MERGED_DIR = '/content/AERIAL_V5'
API_KEY    = 'nso0uHIaz2bKrzlOv7C4'   # ← ضع API Key الخاص بك

if os.path.exists(f'{MERGED_DIR}/data.yaml'):
    n = len(list(Path(f'{MERGED_DIR}/train/images').glob('*')))
    print(f'✅ الداتاسيت موجود: {n:,} صورة في train')

else:
    rf = Roboflow(api_key=API_KEY)

    print('📥 تحميل drones_new v4...')
    ds1 = rf.workspace('tracker-qjlj1').project('drones_new') \
             .version(4).download('yolov8', location='/content/drones_new')

    print('📥 تحميل Airborne v6...')
    ds2 = rf.workspace('airborne-object-detection') \
             .project('airborne-object-detection-4-aod4') \
             .version(6).download('yolov8', location='/content/airborne')

    for split in ['train', 'valid', 'test']:
        os.makedirs(f'{MERGED_DIR}/{split}/images', exist_ok=True)
        os.makedirs(f'{MERGED_DIR}/{split}/labels', exist_ok=True)

    def build_class_map(yaml_path):
        """يقرأ أسماء الكلاسات ويعمل mapping تلقائي لـ 4 كلاسات"""
        with open(yaml_path) as f:
            info = yaml.safe_load(f)
        names = info.get('names', [])
        mapping = {}
        for i, name in enumerate(names):
            n = name.lower()
            if   'bird'     in n:                          mapping[i] = 3
            elif 'heli'     in n:                          mapping[i] = 2
            elif 'plane'    in n or 'airplane' in n \
              or 'aircraft' in n:                          mapping[i] = 1
            elif 'drone'    in n or 'uav' in n \
              or 'quadcopter' in n:                        mapping[i] = 0
            # أي كلاس آخر → لا يُضاف (None)
        return mapping, names

    def copy_dataset(src_root, prefix, class_map):
        """ينسخ الصور مع إعادة تسمية الكلاسات"""
        total = 0
        skipped = 0
        for split in ['train', 'valid', 'val', 'test']:
            img_src = Path(src_root) / split / 'images'
            lbl_src = Path(src_root) / split / 'labels'
            dst_key = 'valid' if split == 'val' else split
            if not img_src.exists(): continue
            img_dst = Path(MERGED_DIR) / dst_key / 'images'
            lbl_dst = Path(MERGED_DIR) / dst_key / 'labels'
            for img in img_src.iterdir():
                if img.suffix.lower() not in ['.jpg','.jpeg','.png']: continue
                lbl = lbl_src / (img.stem + '.txt')
                if not lbl.exists(): continue
                new_lines = []
                with open(lbl) as f:
                    for line in f:
                        parts = line.strip().split()
                        if not parts: continue
                        new_cls = class_map.get(int(parts[0]))
                        if new_cls is None: continue
                        parts[0] = str(new_cls)
                        new_lines.append(' '.join(parts))
                if not new_lines:
                    skipped += 1
                    continue
                uid = f'{prefix}_{img.stem}'
                shutil.copy2(img, img_dst / (uid + img.suffix))
                with open(lbl_dst / (uid + '.txt'), 'w') as f:
                    f.write('\n'.join(new_lines))
                total += 1
        return total, skipped

    print('\n🔀 دمج الداتاسيتات...')
    for ds, prefix, label in [
        (ds1, 'd1', 'drones_new'),
        (ds2, 'd2', 'airborne'),
    ]:
        cmap, orig = build_class_map(Path(ds.location) / 'data.yaml')
        print(f'  {label}: {orig}')
        print(f'    mapping: {cmap}')
        copied, skipped = copy_dataset(ds.location, prefix, cmap)
        print(f'    ✅ {copied:,} صورة | ⏭️ {skipped} محذوفة')

    with open(f'{MERGED_DIR}/data.yaml', 'w') as f:
        yaml.dump({
            'path'  : MERGED_DIR,
            'train' : 'train/images',
            'val'   : 'valid/images',
            'test'  : 'test/images',
            'nc'    : 4,
            'names' : ['drone', 'airplane', 'helicopter', 'bird']
        }, f)

    n = len(list(Path(f'{MERGED_DIR}/train/images').glob('*')))
    print(f'\n✅ تم الدمج! {n:,} صورة في train')

print(f'📂 الداتاسيت جاهز: {MERGED_DIR}')

📥 تحميل drones_new v4...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drones_new in yolov8:: 100%|██████████| 46008/46008 [00:10<00:00, 4406.46it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📥 تحميل Airborne v6...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/airborne in yolov8:: 100%|██████████| 45044/45044 [00:06<00:00, 7167.02it/s]



🔀 دمج الداتاسيتات...
  drones_new: ['drone']
    mapping: {0: 0}
    ✅ 22,805 صورة | ⏭️ 193 محذوفة
  airborne: ['airplane', 'bird', 'drone', 'helicopter']
    mapping: {0: 1, 1: 3, 2: 0, 3: 2}
    ✅ 21,920 صورة | ⏭️ 596 محذوفة

✅ تم الدمج! 35,413 صورة في train
📂 الداتاسيت جاهز: /content/AERIAL_V5


In [ ]:

# التحقق من البيانات
import os

DATA_YAML = '/content/AERIAL_V5/data.yaml'

print("="*50)
print("📁 التحقق من مجلد AERIAL_V5")
print("="*50)

# 1. هل المجلد موجود؟
if os.path.exists('/content/AERIAL_V5'):
    print("✅ مجلد AERIAL_V5 موجود")
else:
    print("❌ مجلد AERIAL_V5 غير موجود")
    print("   شغل Cell 3 أولاً")

# 2. عرض محتويات المجلد
print("\n📂 محتويات AERIAL_V5:")
if os.path.exists('/content/AERIAL_V5'):
    for item in os.listdir('/content/AERIAL_V5'):
        path = f'/content/AERIAL_V5/{item}'
        if os.path.isdir(path):
            num_files = len(os.listdir(path)) if os.path.exists(path) else 0
            print(f"   📁 {item}/ : {num_files} عنصر")
        else:
            print(f"   📄 {item}")

# 3. التحقق من data.yaml
print("\n📄 محتوى data.yaml:")
if os.path.exists(DATA_YAML):
    with open(DATA_YAML, 'r') as f:
        print(f.read())
else:
    print("❌ data.yaml غير موجود")

# 4. التحقق من وجود الصور والـ labels
print("\n🖼️ التحقق من الصور:")
for split in ['train', 'valid']:
    img_path = f'/content/AERIAL_V5/{split}/images'
    if os.path.exists(img_path):
        num_imgs = len(os.listdir(img_path))
        print(f"   {split}/images: {num_imgs} صورة")
    else:
        print(f"   ❌ {split}/images غير موجود")

print("\n🏷️ التحقق من الـ labels:")
for split in ['train', 'valid']:
    lbl_path = f'/content/AERIAL_V5/{split}/labels'
    if os.path.exists(lbl_path):
        num_lbls = len(os.listdir(lbl_path))
        print(f"   {split}/labels: {num_lbls} ملف")
    else:
        print(f"   ❌ {split}/labels غير موجود")

print("\n" + "="*50)
print("✅ إذا ظهرت جميع المؤشرات إيجابية، يمكنك بدء التدريب")
print("="*50)

📁 التحقق من مجلد AERIAL_V5
✅ مجلد AERIAL_V5 موجود

📂 محتويات AERIAL_V5:
   📁 train/ : 2 عنصر
   📄 data.yaml
   📁 test/ : 2 عنصر
   📁 valid/ : 2 عنصر

📄 محتوى data.yaml:
names:
- drone
- airplane
- helicopter
- bird
nc: 4
path: /content/AERIAL_V5
test: test/images
train: train/images
val: valid/images


🖼️ التحقق من الصور:
   train/images: 35413 صورة
   valid/images: 6065 صورة

🏷️ التحقق من الـ labels:
   train/labels: 35413 ملف
   valid/labels: 6065 ملف

✅ إذا ظهرت جميع المؤشرات إيجابية، يمكنك بدء التدريب


In [ ]:

# Cell 5: ابدأ التدريب الآن
from ultralytics import YOLO
import torch, shutil, os

EPOCHS    = 100
IMG_SIZE  = 640
BATCH     = 16
SAVE_DIR  = '/content/drive/MyDrive/drone_training'
RUN_NAME  = 'yolo11m_aerial_v5'
DATA_YAML = '/content/AERIAL_V5/data.yaml'

RUN_PATH   = f'{SAVE_DIR}/{RUN_NAME}'
LAST_PT    = f'{RUN_PATH}/weights/last.pt'
BEST_PT    = f'{RUN_PATH}/weights/best.pt'
DRIVE_LAST = f'{SAVE_DIR}/last_v5.pt'
DRIVE_BEST = f'{SAVE_DIR}/yolo11m_aerial_best_v5.pt'

USE_GPU = torch.cuda.is_available()
print('='*55)
print('🚀 بدء تدريب yolo11m على AERIAL_V5')
print('='*55)
print(f'الجهاز      : {torch.cuda.get_device_name(0) if USE_GPU else "CPU ⚠️"}')
print(f'صور التدريب : 35,413')
print(f'صور التقييم : 6,065')
print(f'الكلاسات    : drone | airplane | helicopter | bird')
print(f'Epochs      : {EPOCHS}')
print(f'ImgSize     : {IMG_SIZE}')
print(f'Batch       : {BATCH}')
print(f'الوقت المتوقع: ~8-10 ساعات')
print('='*55)
print('💾 سيتم حفظ checkpoint كل epoch في Drive')
print('🔄 يمكنك إيقاف التشغيل واستئنافه لاحقاً')
print('='*55)

def save_to_drive(trainer):
    src = f'{trainer.save_dir}/weights/last.pt'
    if os.path.exists(src):
        shutil.copy(src, DRIVE_LAST)
        print(f'💾 [Epoch {trainer.epoch+1}/{EPOCHS}] حفظ في Drive ✅')

resume_from = None
if os.path.exists(LAST_PT):
    resume_from = LAST_PT
    print(f'\n🔄 استئناف من checkpoint محلي')
elif os.path.exists(DRIVE_LAST):
    os.makedirs(f'{RUN_PATH}/weights', exist_ok=True)
    shutil.copy(DRIVE_LAST, LAST_PT)
    resume_from = LAST_PT
    print(f'\n🔄 استئناف من Drive')
else:
    print('\n🆕 بدء تدريب من الصفر')

# بدء التدريب
if resume_from:
    model = YOLO(resume_from)
    model.add_callback('on_train_epoch_end', save_to_drive)
    results = model.train(resume=True)
else:
    model = YOLO('yolo11m.pt')
    model.add_callback('on_train_epoch_end', save_to_drive)
    results = model.train(
        data          = DATA_YAML,
        epochs        = EPOCHS,
        imgsz         = IMG_SIZE,
        batch         = BATCH,
        device        = 0,

        patience      = 20,
        mosaic        = 1.0,
        mixup         = 0.3,      # تحسين للدrones الصغيرة
        copy_paste    = 0.3,      # مفيد للأجسام الصغيرة
        scale         = 0.5,
        degrees       = 15,
        flipud        = 0.1,
        fliplr        = 0.5,

        optimizer     = 'AdamW',
        lr0           = 0.001,
        warmup_epochs = 3,

        project       = SAVE_DIR,
        name          = RUN_NAME,
        exist_ok      = True,
        save          = True,
        save_period   = 1,
    )

# حفظ النموذج النهائي
print('\n💾 حفظ النموذج النهائي في Drive...')
if os.path.exists(BEST_PT):
    shutil.copy(BEST_PT, DRIVE_BEST)
    print(f'   ✅ yolo11m_aerial_best_v5.pt')
if os.path.exists(LAST_PT):
    shutil.copy(LAST_PT, DRIVE_LAST)
    print(f'   ✅ last_v5.pt')

# عرض النتائج
try:
    map50   = results.results_dict.get('metrics/mAP50(B)', 0)
    map5095 = results.results_dict.get('metrics/mAP50-95(B)', 0)
    print('\n' + '='*55)
    print('📊 النتائج النهائية:')
    print(f'   mAP@50    : {map50:.3f}')
    print(f'   mAP@50-95 : {map5095:.3f}')
    if map50 >= 0.85:
        print('🏆 ممتاز! النموذج جاهز للاستخدام')
    elif map50 >= 0.70:
        print('✅ جيد جداً')
    elif map50 >= 0.60:
        print('👍 جيد - يمكن استخدامه')
    else:
        print('⚠️ يحتاج تحسين - زد epochs أو حسّن البيانات')
    print('='*55)
except:
    pass

print('\n✅ التدريب اكتمل!')
print(f'📁 أفضل نموذج: {DRIVE_BEST}')
print('\n🔄 للاستئناف لاحقاً: شغل Cell 5 مرة أخرى')

🚀 بدء تدريب yolo11m على AERIAL_V5
الجهاز      : Tesla T4
صور التدريب : 35,413
صور التقييم : 6,065
الكلاسات    : drone | airplane | helicopter | bird
Epochs      : 100
ImgSize     : 640
Batch       : 16
الوقت المتوقع: ~8-10 ساعات
💾 سيتم حفظ checkpoint كل epoch في Drive
🔄 يمكنك إيقاف التشغيل واستئنافه لاحقاً

🔄 استئناف من checkpoint محلي
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/AERIAL_V5/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hs